In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

In [2]:
import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from models.lstm_model import LSTM_Model
from utils import EarlyStopping, get_device, EpochTrainer, build_seq, Get_yf_data
from operator import itemgetter
from sklearn.preprocessing import MinMaxScaler

# Parameters

In [3]:
model_output_filename = "lstm_checkpoint"

## model config

In [4]:
batch_size = 100
epochs = 100
hidden_size = 64
num_output = 1
num_layers = 2
dropout = 0.2
bidirectional = False
patience = 5
test_size = 0.2
lr = 0.001

In [5]:
seq_len = 30
data_len = 600


eval_method = 'RMSE'
criterion = nn.MSELoss()

In [6]:
input_cols = ["Open", "High", "Low", "Close", "Volume"]
identifier_cols = ['Date']
target_cols = ['Close']

# Get Sample Data

## Dataset additional Parameters

In [7]:
steps_ahead = 5
ticker = "VOO"
start = "2015-01-01"
end = "2026-04-29"

## Load data from yfinance

In [8]:
data = Get_yf_data( ticker , start , end)

[*********************100%***********************]  1 of 1 completed


# Data Processing

## Scale inputs and target

In [9]:
x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X_all = x_scaler.fit_transform(data[input_cols].values)
y_all = y_scaler.fit_transform(data[target_cols].values)

## Build Sequence

In [10]:
id_all = data[identifier_cols].values.flatten().tolist()

Seqs = build_seq( seq_len , X_all, steps_ahead, id_all , y_all  )

X_seq, y_seq, X_id, y_id = itemgetter('X_Seq', 'y_Seq', 'X_id', 'y_id')(Seqs)

In [11]:
X_seq, y_seq, X_id, y_id = X_seq[-data_len:], y_seq[-data_len:], X_id[-data_len:], y_id[-data_len:]

## Convert to Tensor

In [12]:
X_tensor = torch.from_numpy(np.asarray(X_seq, dtype=np.float32))
y_tensor = torch.tensor(np.array(y_seq), dtype=torch.float32)

In [13]:
full_dataset = TensorDataset(X_tensor, y_tensor)

## Split data into training and testing

In [14]:
input_size = X_tensor.shape[-1]

In [15]:
test_records = int(test_size * len(full_dataset))
train_records = len(full_dataset) - test_records

train_dataset, test_dataset = random_split(
    full_dataset,
    [train_records, test_records],
    generator=torch.Generator().manual_seed(42),
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


# Prepare Model

## Prepare config of the model

In [16]:
preprocess_config = {
    "seq_length": seq_len,
    "input_size": input_size,
    "task_type": "classification",
}

In [17]:
model_config = {
    "input_size": input_size,
    "hidden_size": hidden_size,
    "num_output": num_output,
    "num_layers": num_layers,
    "dropout": dropout,
    "bidirectional": bidirectional,
}

## Get Model

In [18]:
device = get_device()

In [19]:
model = LSTM_Model(
    input_size=input_size,
    hidden_size=hidden_size,
    num_output=num_output,
    num_layers=num_layers,
    dropout=dropout,
    bidirectional=bidirectional,
).to(device)

In [20]:
optimizer = torch.optim.Adam(model.parameters(), lr = lr)

## Early stopping and checkpoint setup

In [21]:
early_stopping = EarlyStopping(
    patience=patience,
    path=f"../checkpoints/{model_output_filename}.pt",
    checkpoint_data={
        "model_config": model_config,
        "preprocess_config": preprocess_config,
        "target_cols": target_cols,
        "input_cols": input_cols,
        "x_scaler": x_scaler,
        "y_scaler": y_scaler,
        "steps_ahead" : steps_ahead,
    },
)

# Loop epochs and train model

In [22]:
epoch_trainer = EpochTrainer(
    model = model,
    early_stopping = early_stopping,
    device = device,
    optimizer = optimizer,
    criterion = criterion,
    eval_method = eval_method
)

In [23]:
for epoch in range(epochs):

    avg_train_loss, avg_val_loss, result = epoch_trainer(train_loader , test_loader )

    for key, value in result.items():        
        
        print(f"Epoch {epoch + 1:3d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | {key}: {value:.4f}")
    
    # ==================== Early Stopping Check ====================

    early_stopping(avg_val_loss, model)

    if early_stopping.early_stop:
        
        print("Early stopping triggered! Training stopped.")
        
        break
        

Epoch   1 | Train Loss: 0.4339 | Val Loss: 0.2832 | RMSE: 0.5322
Epoch   2 | Train Loss: 0.2057 | Val Loss: 0.0477 | RMSE: 0.2183
Epoch   3 | Train Loss: 0.0435 | Val Loss: 0.0681 | RMSE: 0.2609
Epoch   4 | Train Loss: 0.0346 | Val Loss: 0.0101 | RMSE: 0.1003
Epoch   5 | Train Loss: 0.0232 | Val Loss: 0.0233 | RMSE: 0.1528
Epoch   6 | Train Loss: 0.0254 | Val Loss: 0.0106 | RMSE: 0.1030
Epoch   7 | Train Loss: 0.0143 | Val Loss: 0.0128 | RMSE: 0.1132
Epoch   8 | Train Loss: 0.0164 | Val Loss: 0.0129 | RMSE: 0.1136
Epoch   9 | Train Loss: 0.0139 | Val Loss: 0.0082 | RMSE: 0.0904
Epoch  10 | Train Loss: 0.0130 | Val Loss: 0.0088 | RMSE: 0.0938
Epoch  11 | Train Loss: 0.0129 | Val Loss: 0.0077 | RMSE: 0.0878
Epoch  12 | Train Loss: 0.0118 | Val Loss: 0.0084 | RMSE: 0.0917
Epoch  13 | Train Loss: 0.0108 | Val Loss: 0.0075 | RMSE: 0.0865
Epoch  14 | Train Loss: 0.0111 | Val Loss: 0.0068 | RMSE: 0.0827
Epoch  15 | Train Loss: 0.0103 | Val Loss: 0.0065 | RMSE: 0.0808
Epoch  16 | Train Loss: 0